In [1]:
import os
import re
import glob

import numpy as np
import pandas as pd

In [4]:
import os
import re
import glob
import pandas as pd


DATASETS = [
    "gy_0", "gy_2", "gy_4", "gy_5", "gy_7", "gy_11_mut_5", "gy_13_maybe_mut_20",
    "gy_14_more_iter", "j_1", "gy_2nd_iter_1", "gy_0_test",
    "gy_2nd_iter_try_23_balm_deployed",
    "gy_2nd_iter_try_24_control_interface_fixing",
    "gy_2nd_iter_try_29_balm_deployed_weight_10_length_15",
    "gy_2nd_iter_try_31_control_length_15",
    "gy_2nd_iter_try_39_fixed_animations",
    "gy_2nd_iter_try_40_fixed_new_filters",
    "gy_2nd_iter_try_41_more_iter",
]
# de-duplicate while preserving order
DATASETS = list(dict.fromkeys(DATASETS))

ROOT = "/FastHome/gyula/designs/place_holder/DATA"

THREE_TO_ONE = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C",
    "GLN": "Q", "GLU": "E", "GLY": "G", "HIS": "H", "ILE": "I",
    "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P",
    "SER": "S", "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V",
    "MSE": "M", "SEC": "U", "PYL": "O",
}

RANKED_COLS = ["rank_norm", "rank_raw", "Design", "pdb_path", "found_in", "Target", "proteina"]


def norm_rank(x):
    m = re.search(r"(\d+)", str(x))
    return str(int(m.group(1))) if m else None


def norm_design(s):
    """Normalize a Design key to a plain string, stripping a trailing '.0'
    that shows up when pandas infers a numeric-looking column as float64."""
    s = s.astype(str)
    return s.str.replace(r"\.0$", "", regex=True)


def get_chain_sequence(pdb_path, chain_id):
    """Extract the one-letter amino-acid sequence for a given chain from a PDB file,
    reading one residue per CA atom in file order. Returns None if the chain/file
    isn't found."""
    seq = []
    seen_res = set()
    try:
        with open(pdb_path) as f:
            for line in f:
                if not (line.startswith("ATOM") or line.startswith("HETATM")):
                    continue
                if line[12:16].strip() != "CA":
                    continue
                if line[21].strip() != chain_id:
                    continue
                resseq_key = line[22:27]  # resnum + insertion code
                if resseq_key in seen_res:
                    continue
                seen_res.add(resseq_key)
                resname = line[17:20].strip()
                seq.append(THREE_TO_ONE.get(resname, "X"))
    except (FileNotFoundError, OSError):
        return None
    return "".join(seq) if seq else None


def parse_ranked_folder(folder, source):
    rows = []
    for pdb in glob.glob(os.path.join(folder, "*.pdb")):
        base = os.path.basename(pdb)
        m = re.match(r"^(\d+)_(.+?)_model\d+\.pdb$", base)
        if not m:
            continue
        rows.append({
            "rank_norm": norm_rank(m.group(1)),
            "rank_raw": m.group(1),
            "Design": m.group(2),
            "pdb_path": pdb,
            "found_in": source,
            "Target": get_chain_sequence(pdb, "A"),
            "proteina": get_chain_sequence(pdb, "B"),
        })
    out = pd.DataFrame(rows, columns=RANKED_COLS)
    out["Design"] = norm_design(out["Design"])
    return out


def load_mmpbsa(path, suffix):
    """Load an MMPBSA summary. Any column the CSV doesn't have (commonly
    dG_binding_IE / dG_binding_IE_err) comes back as an all-NA column instead
    of raising KeyError. delta_total is the only one that really matters."""
    cols = [
        "job_norm",
        f"delta_total_{suffix}",
        f"dG_binding_IE_{suffix}",
        f"dG_binding_IE_err_{suffix}",
    ]
    if not os.path.isfile(path):
        return pd.DataFrame(columns=cols)

    d = pd.read_csv(path).copy()

    if "subdir" in d.columns:
        d["job_norm"] = d["subdir"].astype(str).str.extract(r"job_(\d+)")[0].map(norm_rank)
    else:
        print(f"  Note: no 'subdir' column in {path} — job mapping will be empty.")
        d["job_norm"] = pd.NA

    d = d.rename(columns={
        "delta_total": f"delta_total_{suffix}",
        "dG_binding_IE": f"dG_binding_IE_{suffix}",
        "dG_binding_IE_err": f"dG_binding_IE_err_{suffix}",
    })

    missing = [c for c in cols if c not in d.columns]
    if missing:
        print(f"  Note: {path} missing {missing} — filling with NA.")
    for c in missing:
        d[c] = pd.NA

    return d[cols].copy()


def empty_lookup(cols):
    return pd.DataFrame(columns=cols, index=pd.Index([], name="Design", dtype=object))


for dataset in DATASETS:
    print(f"\n=== {dataset} ===")

    STATS_CSV = f"{ROOT}/{dataset}/mpnn_design_stats.csv"

    RANKED_ACCEPTED = f"{ROOT}/{dataset}/Accepted/Ranked/"
    RANKED_REJECTED = f"{ROOT}/{dataset}_rejected/Ranked/Ranked/"

    MMPBSA_ACCEPTED = f"{ROOT}/{dataset}/md_sim_analysis/mmpbsa/jobs/delta_total_summary.csv"
    MMPBSA_REJECTED_DIR = f"{ROOT}/{dataset}_rejected/md_sim_analysis/mmpbsa/jobs/"
    MMPBSA_REJECTED = os.path.join(MMPBSA_REJECTED_DIR, os.path.basename(MMPBSA_ACCEPTED))

    OUTPUT_CSV = f"{ROOT}/{dataset}/mpnn_design_stats_augmented.csv"

    # ── 1. Load MPNN stats (guard: stats CSV may not exist) ───────────────────
    if os.path.isfile(STATS_CSV):
        df = pd.read_csv(STATS_CSV, dtype={"Design": str}).copy()
        if "Design" not in df.columns:
            df = df.rename(columns={df.columns[0]: "Design"})
        df["Design"] = norm_design(df["Design"])
    else:
        print(f"  Note: stats CSV not found ({STATS_CSV}) — building from PDB files only.")
        df = None

    # ── 2. Parse Ranked folders separately ────────────────────────────────────
    acc_ranked = (parse_ranked_folder(RANKED_ACCEPTED, "accepted")
                  if os.path.isdir(RANKED_ACCEPTED) else pd.DataFrame(columns=RANKED_COLS))
    has_rejected_folder = os.path.isdir(RANKED_REJECTED)
    rej_ranked = (parse_ranked_folder(RANKED_REJECTED, "rejected")
                  if has_rejected_folder else pd.DataFrame(columns=RANKED_COLS))

    if not has_rejected_folder:
        print(f"  Note: rejected folder not found ({RANKED_REJECTED}) — accepted only.")

    # Index by rank within each branch
    acc_ranked = acc_ranked.drop_duplicates(subset=["rank_norm"], keep="first")
    rej_ranked = rej_ranked.drop_duplicates(subset=["rank_norm"], keep="first")

    # If there was no stats CSV, build the base dataframe from the PDB Designs
    if df is None:
        all_designs = sorted(set(acc_ranked["Design"].dropna()) | set(rej_ranked["Design"].dropna()))
        df = pd.DataFrame({"Design": all_designs})
        if df.empty:
            print("  Nothing to do: no stats CSV and no ranked PDBs. Skipping.")
            continue
        df["Design"] = norm_design(df["Design"])
        print(f"  Built base dataframe with {len(df)} designs from PDB files.")

    # ── 3. Load MMPBSA summaries ──────────────────────────────────────────────
    acc_mmpbsa = load_mmpbsa(MMPBSA_ACCEPTED, "accepted")
    has_rejected_mmpbsa = os.path.isfile(MMPBSA_REJECTED)
    rej_mmpbsa = load_mmpbsa(MMPBSA_REJECTED, "rejected")

    if not has_rejected_mmpbsa:
        print(f"  Note: rejected MMPBSA summary not found ({MMPBSA_REJECTED}) — accepted only.")

    # ── 4. Map each job to the corresponding ranked PDB in the same branch ─────
    if not acc_mmpbsa.empty:
        acc_job_to_design = acc_mmpbsa.merge(
            acc_ranked, left_on="job_norm", right_on="rank_norm", how="left"
        )[["job_norm", "Design"]].drop_duplicates(subset=["job_norm"], keep="first")
        acc_job_to_design["Design"] = norm_design(acc_job_to_design["Design"])
    else:
        acc_job_to_design = pd.DataFrame(columns=["job_norm", "Design"])

    if not rej_mmpbsa.empty:
        rej_job_to_design = rej_mmpbsa.merge(
            rej_ranked, left_on="job_norm", right_on="rank_norm", how="left"
        )[["job_norm", "Design"]].drop_duplicates(subset=["job_norm"], keep="first")
        rej_job_to_design["Design"] = norm_design(rej_job_to_design["Design"])
    else:
        rej_job_to_design = pd.DataFrame(columns=["job_norm", "Design"])

    # ── 5. Convert job mapping into design-keyed MMPBSA tables ────────────────
    acc_mmpbsa = acc_mmpbsa.merge(acc_job_to_design, on="job_norm", how="left")
    rej_mmpbsa = rej_mmpbsa.merge(rej_job_to_design, on="job_norm", how="left")

    ACC_VAL_COLS = ["delta_total_accepted", "dG_binding_IE_accepted", "dG_binding_IE_err_accepted"]
    REJ_VAL_COLS = ["delta_total_rejected", "dG_binding_IE_rejected", "dG_binding_IE_err_rejected"]

    if not acc_mmpbsa.empty:
        acc_lookup = (acc_mmpbsa.dropna(subset=["Design"])
                      .drop_duplicates(subset=["Design"], keep="first")
                      .set_index("Design")[ACC_VAL_COLS])
    else:
        acc_lookup = empty_lookup(ACC_VAL_COLS)

    if not rej_mmpbsa.empty:
        rej_lookup = (rej_mmpbsa.dropna(subset=["Design"])
                      .drop_duplicates(subset=["Design"], keep="first")
                      .set_index("Design")[REJ_VAL_COLS])
    else:
        rej_lookup = empty_lookup(REJ_VAL_COLS)

    # ── 6. Merge into the main MPNN dataframe by exact Design ─────────────────
    df = df.merge(acc_lookup, on="Design", how="left")
    df = df.merge(rej_lookup, on="Design", how="left")

    # ── 6b. Merge Target (chain A) / proteina (chain B) sequences ─────────────
    if not acc_ranked.empty:
        acc_seq_lookup = (acc_ranked.drop_duplicates(subset=["Design"], keep="first")
                          .set_index("Design")[["Target", "proteina"]]
                          .rename(columns={"Target": "Target_accepted",
                                           "proteina": "proteina_accepted"}))
    else:
        acc_seq_lookup = empty_lookup(["Target_accepted", "proteina_accepted"])

    if not rej_ranked.empty:
        rej_seq_lookup = (rej_ranked.drop_duplicates(subset=["Design"], keep="first")
                          .set_index("Design")[["Target", "proteina"]]
                          .rename(columns={"Target": "Target_rejected",
                                           "proteina": "proteina_rejected"}))
    else:
        rej_seq_lookup = empty_lookup(["Target_rejected", "proteina_rejected"])

    df = df.merge(acc_seq_lookup, on="Design", how="left")
    df = df.merge(rej_seq_lookup, on="Design", how="left")

    for col in ACC_VAL_COLS + REJ_VAL_COLS + [
        "Target_accepted", "proteina_accepted", "Target_rejected", "proteina_rejected"
    ]:
        if col not in df.columns:
            df[col] = pd.NA

    df["Target"] = df["Target_accepted"].combine_first(df["Target_rejected"])
    df["proteina"] = df["proteina_accepted"].combine_first(df["proteina_rejected"])

    # ── 6c. Sequence lengths ──────────────────────────────────────────────────
    df["Target_length"] = df["Target"].str.len()
    df["proteina_length"] = df["proteina"].str.len()

    # ── 7. Final combined columns ────────────────────────────────────────────
    df["delta_total"] = df["delta_total_accepted"].combine_first(df["delta_total_rejected"])
    df["dG_binding_IE"] = df["dG_binding_IE_accepted"].combine_first(df["dG_binding_IE_rejected"])
    df["dG_binding_IE_err"] = df["dG_binding_IE_err_accepted"].combine_first(
        df["dG_binding_IE_err_rejected"])

    # ── 8. Add accepted/rejected/both label from Ranked folders ──────────────
    acc_designs = set(acc_ranked["Design"].dropna())
    rej_designs = set(rej_ranked["Design"].dropna())

    def source_label(design):
        a = design in acc_designs
        r = design in rej_designs
        if a and r:
            return "both"
        if a:
            return "accepted"
        if r:
            return "rejected"
        return None

    df["found_in"] = df["Design"].map(source_label)

    # ── 9. Save ──────────────────────────────────────────────────────────────
    df = df.copy()
    df["Y"] = df["delta_total"]

    n_y = int(df["Y"].notna().sum())
    if n_y == 0:
        print(f"  WARNING: no delta_total values matched for {dataset} — Y is entirely empty.")
    else:
        print(f"  {n_y}/{len(df)} designs have a delta_total value.")

    df.to_csv(OUTPUT_CSV, index=False)
    print(f"  Saved → {OUTPUT_CSV}")

    preview_cols = [
        "Design", "found_in",
        "delta_total_accepted", "delta_total_rejected", "delta_total",
        "proteina_length", "Target_length",
        "Target", "proteina",
    ]
    preview_cols = [c for c in preview_cols if c in df.columns]
    print(df[preview_cols].head(10).to_string(index=False))


=== gy_0 ===
  Note: rejected folder not found (/FastHome/gyula/designs/place_holder/DATA/gy_0_rejected/Ranked/Ranked/) — accepted only.
  Note: rejected MMPBSA summary not found (/FastHome/gyula/designs/place_holder/DATA/gy_0_rejected/md_sim_analysis/mmpbsa/jobs/delta_total_summary.csv) — accepted only.
  103/545 designs have a delta_total value.
  Saved → /FastHome/gyula/designs/place_holder/DATA/gy_0/mpnn_design_stats_augmented.csv
                    Design found_in  delta_total_accepted delta_total_rejected delta_total  proteina_length  Target_length                                                                                                                                                           Target             proteina
BAX_1F16_l12_s880651_mpnn1      NaN                   NaN                  NaN         NaN              NaN            NaN                                                                                                                                     

In [4]:
df

,Design,Protocol,Length,Seed,Helicity,Target_Hotspot,Sequence,InterfaceResidues,MPNN_score,MPNN_seq_recovery,...,proteina_accepted,Target_rejected,proteina_rejected,Target,proteina,delta_total,dG_binding_IE,dG_binding_IE_err,found_in,Y
0,BAX_1F16_l15_s586146_mpnn1,4stage,15,586146,0.95,74-99,DFMAWKKAAEMIESM,"B1,B2,B3,B4,B5,B6,B7,B8,B9,B10,B11,B12,B13,B15",1.03,0.40,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,BAX_1F16_l15_s586146_mpnn2,4stage,15,586146,0.95,74-99,DFMAWRRAAEMIESM,"B1,B2,B3,B4,B5,B6,B7,B8,B9,B10,B11,B12,B13,B14...",1.05,0.50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,BAX_1F16_l15_s586146_mpnn3,4stage,15,586146,0.95,74-99,DFMAWREAAKMIESM,"B1,B2,B3,B4,B5,B6,B7,B8,B9,B10,B11,B12,B13,B15",1.08,0.40,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,BAX_1F16_l15_s586146_mpnn5,4stage,15,586146,0.95,74-99,DFMAWRRAAELINSM,"B1,B2,B3,B4,B5,B6,B7,B8,B9,B10,B11,B12,B13,B15",1.13,0.60,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,BAX_1F16_l15_s586146_mpnn6,4stage,15,586146,0.95,74-99,DFMAWTEAARMIESM,"B1,B2,B3,B4,B5,B6,B7,B8,B9,B10,B11,B12,B13,B15",1.13,0.40,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
720,BAX_1F16_l15_s693305_mpnn6,4stage,15,693305,0.95,74-99,MEKWREVMNAIVSLY,"B1,B3,B4,B5,B6,B7,B8,B9,B10,B11,B12,B14,B15",1.07,0.14,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
721,BAX_1F16_l15_s845888_mpnn6,4stage,15,845888,0.95,74-99,SIAQKLVEVALSLFV,"B1,B2,B3,B4,B5,B6,B7,B9,B10,B11,B13,B14,B15",1.03,0.36,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
722,BAX_1F16_l15_s693305_mpnn7,4stage,15,693305,0.95,74-99,MEKWRQVMEAIVNMY,"B1,B2,B3,B4,B5,B6,B7,B8,B9,B10,B11,B12,B14,B15",1.07,0.21,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
723,BAX_1F16_l15_s693305_mpnn8,4stage,15,693305,0.95,74-99,MEKWRQVMEHIVSLY,"B1,B2,B3,B4,B5,B6,B7,B8,B9,B10,B11,B12,B14,B15",1.08,0.14,...,MEKWRQVMEHIVSLY,NaN,NaN,MDGSGEQPRGGGPTSSEQIMKTGALLLQGFIQDRAGRMGGEAPELA...,MEKWRQVMEHIVSLY,-84.8,-51.79,6.46,accepted,-84.8


In [5]:
import statsmodels.api as sm

def regress_out(df, confound_col = "Length", target_cols = "Y"):
    X = sm.add_constant(df[confound_col])
    result = df.copy()
    for col in target_cols:
        model = sm.OLS(df[col], X, missing="drop").fit()
        result[col] = model.resid
    return result

df_2 = pd.read_csv("/FastHome/gyula/designs/place_holder/DATA/gy_0_test/mpnn_design_stats_augmented.csv")

df_2 = regress_out(df_2)


df_1 = pd.read_csv("/FastHome/gyula/designs/place_holder/DATA/gy_0/mpnn_design_stats_augmented.csv")


df_1 = regress_out(df_1)

merged = df_1.merge(df_2, on="Design", suffixes=("1", "2"))


In [6]:
import seaborn as sns

sns.scatterplot(merged, x="Y1", y="Y2")
from scipy stats import spearman

import numpy as np
mask = merged[["Y1", "Y2"]].notna().all(axis=1)
print(np.corrcoef(merged.Y1[mask], merged.Y2[mask], type = "spearman"))


SyntaxError: invalid syntax (1735159594.py, line 4)